In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

In [2]:
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)

In [3]:
from pathlib import Path
train_ds, val_ds = tf.keras.utils.audio_dataset_from_directory(
    directory=str(Path.home() / ".cache" / "ml-notebooks" / "speech-commands"),
    batch_size=64,
    output_sequence_length=16000,
    seed=seed,
    validation_split=0.3,
    subset='both'
)

Found 6505 files belonging to 4 classes.


Using 4554 files for training.


Using 1951 files for validation.


In [4]:
label = np.array(train_ds.class_names)
print(label)

['follow' 'learn' 'tree' 'visual']


In [5]:
def squeeze(audio, label):
    audio = tf.squeeze(audio, axis=-1)
    return audio, label

def get_spectogram(waveform):
    spectogram = tf.signal.stft(waveform, frame_length=255, frame_step=128)
    spectogram = tf.abs(spectogram)
    spectogram = spectogram[..., tf.newaxis]
    return spectogram

def make_spectogram_ds(ds):
    return ds.map(
        lambda audio, label: (get_spectogram(audio), label),
        tf.data.AUTOTUNE
    )

In [6]:
train_ds = train_ds.map(squeeze, tf.data.AUTOTUNE)
val_ds = val_ds.map(squeeze, tf.data.AUTOTUNE)

test_ds = val_ds.shard(num_shards=2, index=0)
val_ds = val_ds.shard(num_shards=2, index=1)

In [7]:
train_spectogram_ds = make_spectogram_ds(train_ds)
val_spectogram_ds = make_spectogram_ds(val_ds)
test_spectogram_ds = make_spectogram_ds(test_ds)

In [8]:
for i, j in train_spectogram_ds.take(1):
    break

input_shape = i.shape[1:]
num_label = len(label)

In [9]:
norm_layer = layers.Normalization()
norm_layer.adapt(
    data=train_spectogram_ds.map(lambda i, j: i)
)

In [10]:
model = models.Sequential([
    layers.Input(shape=input_shape),
    layers.Resizing(32, 32),
    norm_layer,
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.5),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_label)
])

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing (Resizing)             │ (None, 32, 32, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalization (Normalization)   │ (None, 32, 32, 1)      │             3 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 30, 30, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 15, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,255,431 (4.79 MB)

 Trainable params: 1,255,428 (4.79 MB)

 Non-trainable params: 3 (16.00 B)

In [12]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [13]:
history = model.fit(
    train_spectogram_ds,
    validation_data=val_spectogram_ds,
    epochs=8,
    callbacks=tf.keras.callbacks.EarlyStopping(verbose=1, patience=2)
)

Epoch 1/8


C:\Users\jorda\Code\ml-notebooks\courses\speech-recognition\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


 1/72 ━━━━━━━━━━━━━━━━━━━━ 59s 838ms/step - accuracy: 0.1719 - loss: 1.4283

 3/72 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.2969 - loss: 1.3741  

 5/72 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.3125 - loss: 1.3400

 7/72 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.3527 - loss: 1.3036

... (226 outputs left out)

65/72 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9310 - loss: 0.2065

68/72 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9313 - loss: 0.2047

71/72 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9313 - loss: 0.2056

72/72 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.9308 - loss: 0.2063 - val_accuracy: 0.9365 - val_loss: 0.1893


In [14]:
for i, (spec_batch, lab_batch) in enumerate(test_spectogram_ds):
    pred = model.predict(spec_batch)

    for j in range(len(lab_batch)):
        t = lab_batch[j].numpy()
        p = np.argmax(pred[j])

        print(f'Batch {i}, Data {j}, ori -> {label[t]}, pred -> {label[p]}')

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step 


Batch 0, Data 0, ori -> tree, pred -> tree
Batch 0, Data 1, ori -> learn, pred -> learn
Batch 0, Data 2, ori -> tree, pred -> tree
Batch 0, Data 3, ori -> follow, pred -> follow
Batch 0, Data 4, ori -> tree, pred -> tree
Batch 0, Data 5, ori -> follow, pred -> follow
Batch 0, Data 6, ori -> follow, pred -> follow
Batch 0, Data 7, ori -> follow, pred -> follow
Batch 0, Data 8, ori -> follow, pred -> follow
Batch 0, Data 9, ori -> learn, pred -> learn
Batch 0, Data 10, ori -> visual, pred -> visual
Batch 0, Data 11, ori -> learn, pred -> learn
Batch 0, Data 12, ori -> tree, pred -> tree
Batch 0, Data 13, ori -> follow, pred -> follow
Batch 0, Data 14, ori -> follow, pred -> learn
Batch 0, Data 15, ori -> tree, pred -> tree
Batch 0, Data 16, ori -> learn, pred -> tree
Batch 0, Data 17, ori -> follow, pred -> follow
Batch 0, Data 18, ori -> visual, pred -> visual
Batch 0, Data 19, ori -> learn, pred -> learn
Batch 0, Data 20, ori -> learn, pred -> learn
Batch 0, Data 21, ori -> tree, pred 

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step


Batch 1, Data 0, ori -> visual, pred -> visual
Batch 1, Data 1, ori -> tree, pred -> tree
Batch 1, Data 2, ori -> learn, pred -> learn
Batch 1, Data 3, ori -> tree, pred -> tree
Batch 1, Data 4, ori -> follow, pred -> follow
Batch 1, Data 5, ori -> visual, pred -> visual
Batch 1, Data 6, ori -> follow, pred -> follow
Batch 1, Data 7, ori -> tree, pred -> visual
Batch 1, Data 8, ori -> visual, pred -> visual
Batch 1, Data 9, ori -> visual, pred -> visual
Batch 1, Data 10, ori -> visual, pred -> visual
Batch 1, Data 11, ori -> follow, pred -> follow
Batch 1, Data 12, ori -> follow, pred -> follow
Batch 1, Data 13, ori -> learn, pred -> learn
Batch 1, Data 14, ori -> learn, pred -> learn
Batch 1, Data 15, ori -> follow, pred -> follow
Batch 1, Data 16, ori -> visual, pred -> learn
Batch 1, Data 17, ori -> follow, pred -> follow
Batch 1, Data 18, ori -> visual, pred -> visual
Batch 1, Data 19, ori -> learn, pred -> learn
Batch 1, Data 20, ori -> tree, pred -> tree
Batch 1, Data 21, ori -> 

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step


... (40 outputs left out)

Batch 14, Data 33, ori -> learn, pred -> learn
Batch 14, Data 34, ori -> visual, pred -> tree
Batch 14, Data 35, ori -> learn, pred -> learn
Batch 14, Data 36, ori -> tree, pred -> tree
Batch 14, Data 37, ori -> visual, pred -> visual
Batch 14, Data 38, ori -> tree, pred -> visual
Batch 14, Data 39, ori -> tree, pred -> tree
Batch 14, Data 40, ori -> visual, pred -> visual
Batch 14, Data 41, ori -> follow, pred -> follow
Batch 14, Data 42, ori -> visual, pred -> visual
Batch 14, Data 43, ori -> visual, pred -> visual
Batch 14, Data 44, ori -> follow, pred -> follow
Batch 14, Data 45, ori -> learn, pred -> learn
Batch 14, Data 46, ori -> learn, pred -> learn
Batch 14, Data 47, ori -> visual, pred -> visual
Batch 14, Data 48, ori -> visual, pred -> visual
Batch 14, Data 49, ori -> learn, pred -> learn
Batch 14, Data 50, ori -> follow, pred -> follow
Batch 14, Data 51, ori -> tree, pred -> tree
Batch 14, Data 52, ori -> learn, pred -> learn
Batch 14, Data 53, ori -> visual, pred -> visual

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step


Batch 15, Data 0, ori -> tree, pred -> tree
Batch 15, Data 1, ori -> visual, pred -> visual
Batch 15, Data 2, ori -> tree, pred -> tree
Batch 15, Data 3, ori -> tree, pred -> tree
Batch 15, Data 4, ori -> tree, pred -> tree
Batch 15, Data 5, ori -> follow, pred -> follow
Batch 15, Data 6, ori -> visual, pred -> visual
Batch 15, Data 7, ori -> visual, pred -> visual
Batch 15, Data 8, ori -> visual, pred -> visual
Batch 15, Data 9, ori -> learn, pred -> learn
Batch 15, Data 10, ori -> visual, pred -> visual
Batch 15, Data 11, ori -> learn, pred -> learn
Batch 15, Data 12, ori -> follow, pred -> follow
Batch 15, Data 13, ori -> learn, pred -> learn
Batch 15, Data 14, ori -> follow, pred -> follow
Batch 15, Data 15, ori -> learn, pred -> learn
Batch 15, Data 16, ori -> tree, pred -> tree
Batch 15, Data 17, ori -> tree, pred -> tree
Batch 15, Data 18, ori -> visual, pred -> visual
Batch 15, Data 19, ori -> visual, pred -> visual
Batch 15, Data 20, ori -> tree, pred -> tree
Batch 15, Data 21